In [0]:
# Define input widgets for the source system, table, timestamp, and file format.
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Build the landing path hierarchy and stop gracefully when no new files are available.
landing_volume_root = "/Volumes/caresync_dbws/landing/landing_volume"
source_system_path = f"{landing_volume_root}/{source_system}"
table_path = f"{source_system_path}/{table_name}"
landing_path = f"{table_path}/{landing_timestamp}"


def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


required_paths = [landing_volume_root, source_system_path, table_path, landing_path]

if not all(path_exists(path) for path in required_paths):
    print("no new files have been received")
    dbutils.notebook.exit("no new files have been received")

In [0]:
# Read the landing files using the format passed through the notebook widgets.
if source_file_format == "csv":
    landing_df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(landing_path)
    )
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
# Preview the landing data before applying bronze-layer transformations.
display(landing_df)

In [0]:
# Add audit metadata columns before writing the records to the bronze table.
from pyspark.sql import functions as F

updated_df = (
    landing_df
    .withColumn("landing_timestamp", F.lit(landing_timestamp))
    .withColumn("insert_timestamp", F.current_timestamp())
)


In [0]:
%sql
-- SQL validation step for checking records already available in the silver hospitals table.
-- Use this query to confirm the downstream structure and sample data before bronze-to-silver comparisons.
select * from caresync_dbws.silver.hospitals

In [0]:
# Append the transformed records into the target bronze Delta table.
updated_df.write.format("delta").mode("append").saveAsTable(f"caresync_dbws.bronze.{table_name}")